[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/00_tag_6_8_optionale_datensaetze_vorbereiten.ipynb)

# Tag 6-8 - 00: Datensätze vorbereiten

Dieses Notebook lädt **CIFAR-10 als Standard** einmalig in `notebooks/Day_6_8/datasets/`. Die Beispiele 09 und 12 verwenden anschließend ausschließlich diesen lokalen Cache und starten keinen Download mehr. Weitere Datensätze lassen sich weiterhin gezielt auswählen.


## Warum ein extra Notebook?

Downloads sind in Kursumgebungen oft langsam, instabil oder durch Firewalls blockiert. Deshalb sollten große Datensätze nicht nebenbei in einem Trainingsnotebook gestartet werden. Hier wird gezielt gecached:

- `datasets/` bleibt lokal und ist in `.gitignore`.
- Jeder Datensatz wird einzeln geladen.
- Wenn ein Download hängt, wird nur dieses Notebook unterbrochen, nicht das eigentliche Beispiel.

Torchvision bietet viele eingebaute Datensätze über `torchvision.datasets`. Die API ist meist ähnlich: `root`, `transform`, `download=True`, plus je nach Datensatz `train`, `split` oder `target_type`.


In [ ]:
from pathlib import Path
import time

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DATA_DIR = Path("datasets")
DATA_DIR.mkdir(exist_ok=True)

print("Datenordner:", DATA_DIR.resolve())
print("PyTorch:", torch.__version__)


## Datensatzkatalog

Gute Kandidaten für den Kurs:

| Name | Aufgabe | Warum sinnvoll? |
| --- | --- | --- |
| `CIFAR10` | 10 Klassen, 32x32 RGB | schnell, Standardbaseline, aber nicht sehr realistisch |
| `CIFAR100` | 100 Klassen, 32x32 RGB | deutlich schwieriger, gut für Hyperparameter-Vergleiche |
| `STL10` | 10 Klassen, 96x96 RGB | größere Bilder als CIFAR, Transfer/Feature Learning interessanter |
| `OxfordIIITPet` | Haustiere, 37 Klassen | realistischere Objektbilder, auch Segmentierungs-Targets verfügbar |
| `Flowers102` | Blumen, 102 Klassen | feinere Klassen, gutes Transfer-Learning-Beispiel |
| `Food101` | Essen, 101 Klassen | größer und realistisch, Download kann dauern |
| `DTD` | Texturen, 47 Klassen | Filter/Feature-Maps sehr gut erklärbar |
| `EuroSAT` | Satellitenbilder, 10 Klassen | andere Domäne als Alltagsbilder |
| `Caltech101` | 101 Objektklassen | klassischer Objekt-Datensatz |

Für sehr große oder lizenz-/accountgebundene Datensätze wie ImageNet, COCO, Cityscapes oder CelebA reicht hier bewusst ein Hinweis. Die lädt man nicht spontan während einer Kursübung.


In [ ]:
common_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
])

DATASET_BUILDERS = {
    "CIFAR10_train": lambda: datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=common_transform),
    "CIFAR10_test": lambda: datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=common_transform),
    "CIFAR100_train": lambda: datasets.CIFAR100(root=DATA_DIR, train=True, download=True, transform=common_transform),
    "CIFAR100_test": lambda: datasets.CIFAR100(root=DATA_DIR, train=False, download=True, transform=common_transform),
    "STL10_train": lambda: datasets.STL10(root=DATA_DIR, split="train", download=True, transform=common_transform),
    "STL10_test": lambda: datasets.STL10(root=DATA_DIR, split="test", download=True, transform=common_transform),
    "OxfordIIITPet_trainval": lambda: datasets.OxfordIIITPet(root=DATA_DIR, split="trainval", download=True, transform=common_transform),
    "OxfordIIITPet_test": lambda: datasets.OxfordIIITPet(root=DATA_DIR, split="test", download=True, transform=common_transform),
    "Flowers102_train": lambda: datasets.Flowers102(root=DATA_DIR, split="train", download=True, transform=common_transform),
    "Flowers102_val": lambda: datasets.Flowers102(root=DATA_DIR, split="val", download=True, transform=common_transform),
    "DTD_train": lambda: datasets.DTD(root=DATA_DIR, split="train", download=True, transform=common_transform),
    "DTD_val": lambda: datasets.DTD(root=DATA_DIR, split="val", download=True, transform=common_transform),
    "EuroSAT": lambda: datasets.EuroSAT(root=DATA_DIR, download=True, transform=common_transform),
    "Caltech101": lambda: datasets.Caltech101(root=DATA_DIR, download=True, transform=common_transform),
    # Food101 ist groß. Nur starten, wenn Zeit und Netzwerk stabil sind.
    "Food101_train": lambda: datasets.Food101(root=DATA_DIR, split="train", download=True, transform=common_transform),
    "Food101_test": lambda: datasets.Food101(root=DATA_DIR, split="test", download=True, transform=common_transform),
}

print("Verfügbare Schlüssel:")
for key in DATASET_BUILDERS:
    print("-", key)


## CIFAR-10 für die Beispiele 09 und 12 herunterladen

Die Voreinstellung lädt beide offiziellen Splits von CIFAR-10 in `datasets/`. Danach können die Beispiele 09 und 12 offline mit diesen Dateien arbeiten. Für einen anderen Datensatz `DATASET_TO_DOWNLOAD` auf einen Schlüssel aus der Liste setzen.


In [ ]:
DATASET_TO_DOWNLOAD = 'CIFAR10'  # Standard f?r Beispiel 09

start = time.time()
if DATASET_TO_DOWNLOAD == 'CIFAR10':
    cifar10_train = DATASET_BUILDERS['CIFAR10_train']()
    cifar10_test = DATASET_BUILDERS['CIFAR10_test']()
    dataset = cifar10_train  # wird vom Smoke Test unten verwendet
    print('Geladen: CIFAR10 (Train und Test)')
    print('Trainingsbilder:', len(cifar10_train))
    print('Testbilder:', len(cifar10_test))
    print('Klassen:', cifar10_train.classes)
else:
    dataset = DATASET_BUILDERS[DATASET_TO_DOWNLOAD]()
    print('Geladen:', DATASET_TO_DOWNLOAD)
    print('Samples:', len(dataset))
    print('Klassen:', getattr(dataset, 'classes', 'keine classes-Property'))

elapsed = time.time() - start
print('Dauer Sekunden:', round(elapsed, 1))


## Kurzer Smoke Test

Diese Zelle prüft, ob ein geladener Datensatz mit `DataLoader` nutzbar ist. Bei manchen Datensätzen sind Labels einfache Zahlen, bei anderen Tupel. Für die Kursbeispiele brauchen wir meist Klassifikationslabels.


In [ ]:
loader = DataLoader(Subset(dataset, range(min(16, len(dataset)))), batch_size=4, shuffle=True)
batch = next(iter(loader))

if isinstance(batch, (tuple, list)):
    images, targets = batch[0], batch[1]
    print("Bildbatch:", images.shape, images.dtype)
    print("Targets:", targets)
else:
    print(type(batch), batch)
